# Env Setup

In [ ]:
%pip install uv --quiet
!uv pip install iqm-client[qiskit]==29.14 --quiet
%pip install qiskit-aer==0.16.1 qrisp --quiet
%pip install lagrangeclient \
    --index-url https://gitlab.linksfoundation.com/api/v4/projects/1709/packages/pypi/simple \
    --quiet
!lagrangeclient
import json
with open("tokens.json", "r") as f:
    config = json.load(f)
ACCESS_TOKEN = config["access_token"]
URL_PROVIDER = "https://spark.quantum.linksfoundation.com/station"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.9.0+cpu requires sympy>=1.13.3, but you have sympy 1.13.0 which is incompatible.
== DEVICE LOGIN FLOW ==
🔗 Visit this URL in your browser:
https://spark.quantum.linksfoundation.com/auth/re

In [ ]:
from iqm.qiskit_iqm import IQMProvider
from qiskit import QuantumCircuit, transpile
from qiskit.transpiler import CouplingMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error, ReadoutError
import math, random

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def get_bitstring_parity(bitstring) -> int:
    """
    Calculates parity of a bitstring (Sum modulo 2).
    """
    return sum(int(bit) for bit in bitstring) % 2

def get_bases_parity(bases) -> int:
    """
    Returns expected parity based on HBB99 rules:
    - 0 Ys (XXX) -> Even Parity (0)
    - 2 Ys (XYY) -> Odd Parity (1)
    """
    return 1 if bases.count("Y") % 4 == 2 else 0

def majority(counts) -> str:
    """
    Returns the most frequent bitstring from shot counts.
    """
    return max(counts, key=counts.get)

def print_table_header(col_names, widths):
    """
    Prints table header.
    """
    header = "| " + " | ".join(f"{name:<{w}}" for name, w in zip(col_names, widths)) + " |"
    separator = "-" * len(header)
    print(separator)
    print(header)
    print(separator)

def print_table_row(values, widths):
    """
    Prints a table row.
    """
    row = "| " + " | ".join(f"{str(val):<{w}}" for val, w in zip(values, widths)) + " |"
    print(row)

# ==========================================
# GLOBAL SETTINGS
# ==========================================

# Hardware Settings
SHOTS_PER_CIRCUIT = 256

# Protocol Settings
OTP_LENGTH = 8
TOTAL_ROUNDS = OTP_LENGTH * 6
TEST_ROUNDS_PERCENTAGE = 0.5
ERROR_THRESHOLD = 0.25

# Backend Settings
use_lagrange = True

if use_lagrange == True:
    provider = IQMProvider(url=URL_PROVIDER, token=ACCESS_TOKEN)
    backend = provider.get_backend()
else:
    # Hardware timings and coherence in ns
    time_1q = 40
    time_cz = 100
    t1 = 100_000
    t2 = 100_000

    # Error probabilities
    prob_1q = 0.003
    prob_cz = 0.020
    prob_meas = 0.030

    # Thermal relaxation errors
    err_thermal_1q = thermal_relaxation_error(t1, t2, time_1q)
    err_thermal_cz = thermal_relaxation_error(t1, t2, time_cz).expand(
                     thermal_relaxation_error(t1, t2, time_cz))

    # Depolarizing errors
    err_depol_1q = depolarizing_error(prob_1q, 1)
    err_depol_cz = depolarizing_error(prob_cz, 2)

    # Combine thermal and polarizing errors
    err_1q = err_thermal_1q.compose(err_depol_1q)
    err_cz = err_thermal_cz.compose(err_depol_cz)

    # Readout error construction
    err_readout = ReadoutError([
        [1 - prob_meas, prob_meas], # [P(0|0), P(1|0)]
        [prob_meas, 1 - prob_meas]  # [P(0|1), P(1|1)]
    ])

    # Apply to IQM native gates
    iqm_noise_model = NoiseModel()
    iqm_noise_model.add_all_qubit_quantum_error(err_1q, ['id', 'rz', 'sx', 'x'])
    iqm_noise_model.add_all_qubit_quantum_error(err_cz, ['cz'])
    iqm_noise_model.add_all_qubit_readout_error(err_readout)

    # Final setup
    backend = AerSimulator(noise_model=iqm_noise_model)
    iqm_basis_gates = ['id', 'rz', 'sx', 'x', 'cz']
    spark_edges = [
        [0, 2], [2, 0],
        [1, 2], [2, 1],
        [3, 2], [2, 3],
        [4, 2], [2, 4]
    ]
    spark_coupling_map = CouplingMap(spark_edges)

# Demo 1: HBB99 with 3 participants

In [ ]:
# ==========================================
# 0. CONFIGURATION
# ==========================================

QUBIT_LAYOUT = [0, 2, 1]  # Physical mapping: [Alice, Bob, Charlie]

# ==========================================
# 1. CIRCUITS
# ==========================================

def create_circuit(bases):
    qc = QuantumCircuit(3, 3)

    # GHZ State Preparation
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()

    # Participants' Measurements
    for i, basis in enumerate(bases):
        if basis == "Y": qc.sdg(i)
        qc.h(i)
        qc.measure(i, i)

    return qc

# ==========================================
# 2. PROTOCOL SETUP
# ==========================================

def run_protocol(backend):
    print("\n=== 1. INITIALIZATION & EXECUTION ===")

    circuits = {}
    bases_history = []

    # Sifting + Circuits
    for i in range(TOTAL_ROUNDS):
        bases = [random.choice(["X", "Y"]) for _ in range(3)]
        bases_history.append(bases)
        if bases.count("Y") % 2 == 0:
            circuits[i] = create_circuit(bases)

    # Run on Backend
    print("-> Transpiling and sending job...")

    if use_lagrange == True:
        transpiled = transpile(list(circuits.values()), backend,
                            initial_layout=QUBIT_LAYOUT, optimization_level=3)
    else:
        transpiled = transpile(list(circuits.values()), backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # Process Results
    print("-> Results received.")

    result = job.result()
    results = {}
    for idx, round_id in enumerate(circuits.keys()):
        counts = result.get_counts(idx)
        raw_outcome = majority(counts)
        # Reverses to: a b c
        results[round_id] = raw_outcome[::-1]

    # Table Printing
    cols = ["Round", "Bases", "Outcome", "Parity", "Expected"]
    widths = [5, 5, 9, 6, 8]
    print_table_header(cols, widths)
    for i in range(TOTAL_ROUNDS):
        if i in results:
            print_table_row([i, "".join(bases_history[i]), results[i],
                             get_bitstring_parity(results[i]), get_bases_parity(bases_history[i])],
                            widths)
        else:
            print_table_row([i, "".join(bases_history[i]), "-", " ", " "], widths)

    return results, bases_history

# ==========================================
# 3. KEY GEN
# ==========================================

def key_gen(results, bases_history):
    valid_rounds = list(results.keys())
    random.shuffle(valid_rounds)
    total_valid = len(valid_rounds)
    test_count = math.ceil(total_valid * TEST_ROUNDS_PERCENTAGE)
    data_count = total_valid - test_count

    # Check if enough data bits
    if data_count < OTP_LENGTH:
        print(f"\n>>> CRITICAL ERROR: Not enough data rounds to satisfy length.")
        return

    test_rounds_indices = valid_rounds[:test_count]
    data_rounds_indices = valid_rounds[test_count : test_count + OTP_LENGTH]

    # ==========================================
    # 3.1 TEST ROUNDS
    # ==========================================

    print("\n=== 2. TEST ROUNDS ===")

    cols = ["Round", "Expected", "Measured"]
    widths = [5, 8, 8]
    print_table_header(cols, widths)

    good_rounds = 0
    for i in test_rounds_indices:
        measured_parity = get_bitstring_parity(results[i])
        expected_parity = get_bases_parity(bases_history[i])
        print_table_row([i, measured_parity, expected_parity], widths)

        if measured_parity == expected_parity:
            good_rounds += 1
    error_rate = (1 - good_rounds / test_count)

    print(f"\n-> Error Rate: {error_rate:.2%}")

    if error_rate >= ERROR_THRESHOLD:
        print(f"\n>>> CRITICAL ERROR: Potential eavesdropping detected.")
        return

    # ==========================================
    # 3.2 DATA ROUNDS
    # ==========================================

    print("\n=== 3. DATA ROUNDS ===")
    print(f"-> Selected Rounds: {data_rounds_indices}\n")

    alice_key = ""
    bob_share = ""
    charlie_share = ""
    parity_corrections = ""

    for i in data_rounds_indices:
        result = results[i]
        bases = bases_history[i]
        alice_key += result[0]
        bob_share += result[1]
        charlie_share += result[2]
        parity_corrections += str(get_bases_parity(bases))

    print(f"-> Alice's Share:     {alice_key}")
    print(f"-> Bob's Share:       {bob_share}")
    print(f"-> Charlie's Share:   {charlie_share}")
    print(f"-> Parity Correction: {parity_corrections}")

    reconstructed_key = ""
    for i in range(OTP_LENGTH):
        reconstructed_key += str(get_bitstring_parity(bob_share[i] + charlie_share[i] + parity_corrections[i]))

    print(f"\n-> Reconstructed:     {reconstructed_key}")

    if alice_key == reconstructed_key:
        print("\n>>> SUCCESS: Key reconstructed perfectly!")
    else:
        print("\n>>> FAILURE: Mismatch detected.")

# ==========================================
# 4. MAIN
# ==========================================

if __name__ == "__main__":
    try:
        results, bases_history = run_protocol(backend)
        key_gen(results, bases_history)
    except Exception as e:
        print(f"An error occurred: {e}")


=== 1. INITIALIZATION & EXECUTION ===
-> Transpiling and sending job...
-> Results received.
-------------------------------------------------
| Round | Bases | Outcome   | Parity | Expected |
-------------------------------------------------
| 0     | YXX   | -         |        |          |
| 1     | XXX   | 000       | 0      | 0        |
| 2     | YYY   | -         |        |          |
| 3     | YXY   | 001       | 1      | 1        |
| 4     | XYY   | 111       | 1      | 1        |
| 5     | YXY   | 010       | 1      | 1        |
| 6     | YYX   | 010       | 1      | 1        |
| 7     | XYX   | -         |        |          |
| 8     | YXX   | -         |        |          |
| 9     | YYX   | 001       | 1      | 1        |
| 10    | YYY   | -         |        |          |
| 11    | XYY   | 010       | 1      | 1        |
| 12    | YYY   | -         |        |          |
| 13    | YYY   | -         |        |          |
| 14    | XYY   | 001       | 1      | 1        |
| 15  

# Demo 2: Intercept-Resend Attack

In [ ]:
# ==========================================
# 0. CONFIGURATION
# ==========================================

QUBIT_LAYOUT = [0, 2, 1]  # Physical mapping: [Alice, Bob, Charlie]

# ==========================================
# 1. CIRCUITS
# ==========================================

def create_intercept_circuit(alice_basis, charlie_basis):
    qc = QuantumCircuit(3, 3)

    # GHZ State Preparation
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()

    # Alice's Measurement
    if alice_basis == "Y": qc.sdg(0)
    qc.h(0)
    qc.measure(0, 0)
    qc.barrier()

    # Charlie*'s Measurements
    charlies_guess = []
    if random.choice(["X", "Y"]) == "X":
        charlies_guess.append("X")
        if charlie_basis == "X":   charlies_guess.append("X")
        elif charlie_basis == "Y": charlies_guess.append("Y"); qc.sdg([1, 2])
    else:
        charlies_guess.append("Y")
        if charlie_basis == "X":   charlies_guess.append("Y"); qc.sdg(1)
        elif charlie_basis == "Y": charlies_guess.append("X"); qc.sdg(2)
    qc.h([1, 2])
    qc.measure([1, 2], [1, 2])
    charlies_guess.append(charlie_basis)

    return qc, charlies_guess

def create_resend_circuit(expected_basis, expected_outcome, bob_basis):
    qc = QuantumCircuit(1, 1)

    # Charlie*'s State Preparation
    qc.h(0)
    if expected_basis == "Y":
        qc.s(0)
    if expected_outcome == "1":
        qc.z(0)
    qc.barrier()

    # Bob's Measurement
    if bob_basis == "Y":
        qc.sdg(0)
    qc.h(0)
    qc.measure(0, 0)

    return qc

# ==========================================
# 2. PROTOCOL
# ==========================================

def run_protocol(backend):
    print("\n=== 1. INITIALIZATION & EXECUTION ===")

    bases_history = []
    guessed_bases = {}

    # Intercept Phase
    intercept_circuits = {}
    intercept_results = {}

    # -Sifting
    for i in range(TOTAL_ROUNDS):
        bases = [random.choice(["X", "Y"]) for _ in range(3)]
        bases_history.append(bases)
        if bases.count("Y") % 2 == 0:
            intercept_circuits[i], guessed_bases[i] = create_intercept_circuit(bases[0], bases[2])

    # -Run on Backend
    print("-> Transpiling and sending intercept job...")

    if use_lagrange == True:
        transpiled = transpile(list(intercept_circuits.values()), backend,
                            initial_layout=QUBIT_LAYOUT, optimization_level=3)
    else:
        transpiled = transpile(list(intercept_circuits.values()), backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # -Process Results
    print("-> Results received.")

    result = job.result()
    for idx, round_id in enumerate(intercept_circuits.keys()):
        counts = result.get_counts(idx)
        raw_outcome = majority(counts)
        # Reverses to: a b c
        intercept_results[round_id] = raw_outcome[::-1]

    # Resend Phase
    resend_circuits = {}
    resend_results = {}

    # -Circuit Creation
    for i in intercept_results.keys():
        resend_circuits[i] = create_resend_circuit(guessed_bases[i][1],
                                                   intercept_results[i][1],
                                                   bases_history[i][1])

    # -Run on Backend
    print("-> Transpiling and sending resend job...")

    if use_lagrange == True:
        transpiled = transpile(list(resend_circuits.values()), backend, optimization_level=3)
    else:
        transpiled = transpile(list(resend_circuits.values()), backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # -Process Results
    print("-> Results received.")

    result = job.result()
    for idx, round_id in enumerate(resend_circuits.keys()):
        counts = result.get_counts(idx)
        resend_results[round_id] = majority(counts)

    # Table Printing
    cols = ["Round", "Bases", "Intercept", "Guessed Bases", "Resend"]
    widths = [5, 5, 9, 13, 6]
    print_table_header(cols, widths)
    for i in range(TOTAL_ROUNDS):
        if i in intercept_results:
            print_table_row([i, "".join(bases_history[i]), intercept_results[i],
                             "".join(guessed_bases[i]), resend_results[i]], widths)
        else:
            print_table_row([i, "".join(bases_history[i]), "-", " ", " "], widths)

    return intercept_results, bases_history, resend_results, guessed_bases

# ==========================================
# 3. KEY GEN
# ==========================================

def key_gen(intercept_results, bases_history, resend_results, guessed_bases):
    valid_rounds = list(intercept_results.keys())
    random.shuffle(valid_rounds)
    total_valid = len(valid_rounds)
    test_count = math.ceil(total_valid * TEST_ROUNDS_PERCENTAGE)
    data_count = total_valid - test_count

    # Check if enough data bits
    if data_count < OTP_LENGTH:
        print(f"\n>>> CRITICAL ERROR: Not enough data rounds to satisfy length.")
        return

    test_rounds_indices = valid_rounds[:test_count]
    data_rounds_indices = valid_rounds[test_count : test_count + OTP_LENGTH]

    final_results = {}
    for i in intercept_results.keys():
        final_results[i] = intercept_results[i][0] + resend_results[i] + intercept_results[i][2]

    # ==========================================
    # 3.1 TEST ROUNDS
    # ==========================================

    print("\n=== 2. TEST ROUNDS ===")

    cols = ["Round", "Measured", "Expected"]
    widths = [5, 8, 8]
    print_table_header(cols, widths)

    good_rounds = 0
    for i in test_rounds_indices:
        measured_parity = get_bitstring_parity(final_results[i])
        expected_parity = get_bases_parity(bases_history[i])
        print_table_row([i, measured_parity, expected_parity], widths)

        if measured_parity == expected_parity:
            good_rounds += 1
    error_rate = (1 - good_rounds / test_count)

    print(f"\n-> Error Rate: {error_rate:.2%}")

    if error_rate >= ERROR_THRESHOLD:
        print(f"\n>>> CRITICAL ERROR: Potential eavesdropping detected.")
        return
    else:
        print("\n>>> CONCLUSION: Threshold not reached? Charlie* was lucky.")

    # ==========================================
    # 3.2 DATA ROUNDS
    # ==========================================

    print("\n=== 3. DATA ROUNDS ===")
    print(f"-> Selected Rounds: {data_rounds_indices}\n")

    alice_key = ""
    deduced_key = ""

    for i in data_rounds_indices:
        result = final_results[i]
        alice_key += result[0]
        deduced_key += str(get_bitstring_parity(result[1] + result[2] + str(get_bases_parity(guessed_bases[i]))))

    print(f"-> Alice's Key:  {alice_key}")
    print(f"-> Deduced Key:  {deduced_key}")

    if alice_key == deduced_key:
        print("\n>>> SUCCESS: Key deduced perfectly!")
    else:
        print("\n>>> FAILURE: Mismatch detected.")

# ==========================================
# 4. MAIN
# ==========================================

if __name__ == "__main__":
    try:
        intercept_results, bases_history, resend_results, guessed_bases = run_protocol(backend)
        key_gen(intercept_results, bases_history, resend_results, guessed_bases)

    except Exception as e:
        print(f"An error occurred: {e}")


=== 1. INITIALIZATION & EXECUTION ===
-> Transpiling and sending intercept job...
-> Results received.
-> Transpiling and sending resend job...
-> Results received.
------------------------------------------------------
| Round | Bases | Intercept | Guessed Bases | Resend |
------------------------------------------------------
| 0     | XYY   | 111       | XYY           | 1      |
| 1     | XXY   | -         |               |        |
| 2     | XYX   | -         |               |        |
| 3     | XXY   | -         |               |        |
| 4     | YYX   | 011       | XXX           | 1      |
| 5     | XYY   | 101       | YXY           | 0      |
| 6     | YXX   | -         |               |        |
| 7     | YXY   | 001       | XYY           | 0      |
| 8     | YYY   | -         |               |        |
| 9     | XYX   | -         |               |        |
| 10    | XXY   | -         |               |        |
| 11    | YXX   | -         |               |        |
| 12    |

# Demo 3: Fake Entanglement Attack

In [ ]:
# ==========================================
# 0. CONFIGURATION
# ==========================================

GHZ_QUBIT_LAYOUT = [0, 2, 1]  # Physical mapping: [a, b, c]
BS_QUBIT_LAYOUT = [0, 2]  # Physical mapping: [b', c']

# ==========================================
# 1. CIRCUITS
# ==========================================

def create_ghz(alice_basis, charlie_basis):
    ghz = QuantumCircuit(3, 3)

    # State Preparation
    ghz.h(0)
    ghz.cx(0, 1)
    ghz.cx(1, 2)
    ghz.barrier()

    # Alice's Measurement
    if alice_basis == "Y": ghz.sdg(0)
    ghz.h(0)
    ghz.measure(0, 0)
    ghz.barrier()

    # Charlie*'s Measurements
    if alice_basis == "X":
        if charlie_basis == "Y": ghz.sdg([1, 2]);
    elif alice_basis == "Y":
        if charlie_basis == "X":   ghz.sdg(1);
        elif charlie_basis == "Y": ghz.sdg(2);
    ghz.h([1, 2])
    ghz.measure([1, 2], [1, 2])

    return ghz

def create_bs(bob_basis):
    bs = QuantumCircuit(2, 2)

    # State Preparation
    bs.x(0)
    bs.h(0)
    bs.cx(0, 1)
    bs.barrier()

    # Bob's Measurement
    if bob_basis == "Y": bs.sdg(0)
    bs.h(0)
    bs.measure(0, 0)
    bs.barrier()

    # Charlie*'s Measurement
    if bob_basis == "Y": bs.sdg(1)
    bs.h(1)
    bs.measure(1, 1)

    return bs

# ==========================================
# 2. PROTOCOL
# ==========================================

def run_protocol(backend):
    print("\n=== 1. INITIALIZATION & EXECUTION ===")

    ghz_circuits = {}
    bs_circuits = {}
    ghz_results = {}
    bs_results = {}
    bases_history = []

    # Circuits Creation
    for i in range(TOTAL_ROUNDS):
        bases = [random.choice(["X", "Y"]) for _ in range(3)]
        bases_history.append(bases)
        if bases.count("Y") % 2 == 0:
            ghz_circuits[i] = create_ghz(bases[0], bases[2])
            bs_circuits[i] = create_bs(bases[1])

    # GHZ States

    # -Run on Backend
    print("-> Transpiling and sending GHZ job...")

    if use_lagrange == True:
        transpiled = transpile(list(ghz_circuits.values()), backend, initial_layout=GHZ_QUBIT_LAYOUT,
                            optimization_level=3)
    else:
        transpiled = transpile(list(ghz_circuits.values()), backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=GHZ_QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # -Process Results
    print("-> GHZ results received.")

    result = job.result()
    for idx, round_id in enumerate(ghz_circuits.keys()):
        counts = result.get_counts(idx)
        raw_outcome = majority(counts)
        # Reverses to: a b c
        ghz_results[round_id] = raw_outcome[::-1]

    # Bell States

    # -Run on Backend
    print("-> Transpiling and sending BS job...")

    if use_lagrange == True:
        transpiled = transpile(list(bs_circuits.values()), backend, initial_layout=BS_QUBIT_LAYOUT,
                            optimization_level=3)
    else:
        transpiled = transpile(list(bs_circuits.values()), backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=BS_QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # -Process Results
    print("-> BS results received.")

    result = job.result()
    for idx, round_id in enumerate(bs_circuits.keys()):
        counts = result.get_counts(idx)
        raw_outcome = majority(counts)
        # Reverses to: b' c'
        bs_results[round_id] = raw_outcome[::-1]


    # Table Printing
    cols = ["Round", "Bases", "GHZ State", "Bell State"]
    widths = [5, 5, 9, 10]
    print_table_header(cols, widths)
    for i in range(TOTAL_ROUNDS):
        if i in bs_results:
            print_table_row([i, "".join(bases_history[i]), ghz_results[i], bs_results[i]], widths)
        else:
            print_table_row([i, "".join(bases_history[i]), "-", "-"], widths)

    return ghz_results, bs_results, bases_history

# ==========================================
# 3. SIDE FUNCTIONS
# ==========================================

def get_alice_bit(ghz_meas, bases):
    used_bases = []
    if bases[0] == "X":
        used_bases.append("X")
        if bases[2] == "X":   used_bases.append("X"); used_bases.append("X")
        elif bases[2] == "Y": used_bases.append("Y"); used_bases.append("Y")
    elif bases[0] == "Y":
        used_bases.append("Y")
        if bases[2] == "X":   used_bases.append("Y"); used_bases.append("X")
        elif bases[2] == "Y": used_bases.append("X"); used_bases.append("Y")
    alice_bit = str(get_bitstring_parity(ghz_meas[1] + ghz_meas[2] + str(get_bases_parity(used_bases))))

    return alice_bit

def get_bob_bit(bs_meas, bases):
    bob_bit = ""
    charlie_bit = bs_meas[1]
    if bases[1] == "X":
        if charlie_bit == "0":   bob_bit = "1"
        elif charlie_bit == "1": bob_bit = "0"
    elif bases[1] == "Y":
        if charlie_bit == "0":   bob_bit = "0"
        elif charlie_bit == "1": bob_bit = "1"

    return bob_bit

def get_expected_bit(ghz_meas, bs_meas, bases):
    alice_bit = get_alice_bit(ghz_meas, bases)
    bob_bit = get_bob_bit(bs_meas, bases)
    expected_bit = str(alice_bit + bob_bit + str(get_bases_parity(bases)))

    return expected_bit

# ==========================================
# 4. KEY GEN
# ==========================================

def key_gen(ghz_results, bs_results, bases_history):
    valid_rounds = list(ghz_results.keys())
    random.shuffle(valid_rounds)
    total_valid = len(valid_rounds)
    test_count = math.ceil(total_valid * TEST_ROUNDS_PERCENTAGE)
    data_count = total_valid - test_count

    # Check if enough data bits
    if data_count < OTP_LENGTH:
        print(f"\n>>> CRITICAL ERROR: Not enough data rounds to satisfy length.")
        return

    test_rounds_indices = valid_rounds[:test_count]
    data_rounds_indices = valid_rounds[test_count : test_count + OTP_LENGTH]

    # ==========================================
    # 4.1 TEST ROUNDS
    # ==========================================

    print("\n=== 2. TEST ROUNDS ===")

    cols = ["Round", "Measured", "Expected"]
    widths = [5, 8, 8]
    print_table_header(cols, widths)
    good_rounds = 0
    for i in test_rounds_indices:
        final_result = ghz_results[i][0] \
                     + bs_results[i][0]  \
                     + get_expected_bit(ghz_results[i], bs_results[i], bases_history[i])
        measured_parity = get_bitstring_parity(final_result)
        expected_parity = get_bases_parity(bases_history[i])
        print_table_row([i, measured_parity, expected_parity], widths)

        if measured_parity == expected_parity:
            good_rounds += 1
    error_rate = (1 - good_rounds / test_count)

    print(f"\n-> Error Rate: {error_rate:.2%}")

    if error_rate >= ERROR_THRESHOLD:
        print(f"\n>>> CRITICAL ERROR: Potential eavesdropping detected.")
        return

    # ==========================================
    # 4.2 DATA ROUNDS
    # ==========================================

    print("\n=== 3. DATA ROUNDS ===")
    print(f"-> Selected Rounds: {data_rounds_indices}\n")

    alice_key = ""
    deduced_key = ""

    for i in data_rounds_indices:
        alice_key += ghz_results[i][0]
        deduced_key += get_alice_bit(ghz_results[i], bases_history[i])

    print(f"-> Alice's Key:  {alice_key}")
    print(f"-> Deduced Key:  {deduced_key}")

    if alice_key == deduced_key:
        print("\n>>> SUCCESS: Key deduced perfectly!")
    else:
        print("\n>>> FAILURE: Mismatch detected.")

# ==========================================
# 5. MAIN
# ==========================================

if __name__ == "__main__":
    try:
        ghz_results, bs_results, bases_history = run_protocol(backend)
        key_gen(ghz_results, bs_results, bases_history)

    except Exception as e:
        print(f"An error occurred: {e}")


=== 1. INITIALIZATION & EXECUTION ===
-> Transpiling and sending GHZ job...
-> GHZ results received.
-> Transpiling and sending BS job...
-> BS results received.
------------------------------------------
| Round | Bases | GHZ State | Bell State |
------------------------------------------
| 0     | YYX   | 111       | 11         |
| 1     | YXX   | -         | -          |
| 2     | XXX   | 110       | 10         |
| 3     | YXX   | -         | -          |
| 4     | XXY   | -         | -          |
| 5     | YXY   | 100       | 10         |
| 6     | XXY   | -         | -          |
| 7     | XYY   | 111       | 11         |
| 8     | YXY   | 010       | 10         |
| 9     | XYY   | 001       | 11         |
| 10    | XXY   | -         | -          |
| 11    | XXY   | -         | -          |
| 12    | YYX   | 010       | 11         |
| 13    | XYX   | -         | -          |
| 14    | XXX   | 011       | 01         |
| 15    | XXX   | 110       | 10         |
| 16    | XXX   | 00

# Demo 4: Ancilla Attack

In [ ]:
# ==========================================
# 0. CONFIGURATION
# ==========================================

QUBIT_LAYOUT = [0, 2, 1, 3]  # Physical mapping: [Alice, Bob, Charlie, Ancilla]

# ==========================================
# 1. CIRCUITS
# ==========================================

def create_circuit_interception(bases):
    qc = QuantumCircuit(4, 4)

    # GHZ State Preparation
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()

    # Charlie*'s Interception
    qc.h(1)
    qc.cx(1, 3)
    qc.barrier()

    # Alice's and Bob's Measurements
    for i, basis in enumerate(bases[:2]):
        if basis == "Y": qc.sdg(i)
        qc.h(i)
        qc.measure(i, i)
    qc.barrier()

    return qc

def use_as_test_round(circuit, bases):
    circuit.cx(2, 3)
    if bases[:2].count("Y") == 1: circuit.sdg(2)
    circuit.h(2)
    circuit.measure([2, 3], [2, 3])

def use_as_data_round(circuit, bases):
    if bases[0] == "Y": circuit.sdg(2)
    circuit.h(2)
    circuit.measure([2, 3], [2, 3])


# ==========================================
# 2. PROTOCOL
# ==========================================

def sifting():
    print("\n=== 1. INITIALIZATION ===")

    circuits = {}
    bases_history = []

    # Sifting
    for i in range(TOTAL_ROUNDS):
        bases = [random.choice(["X", "Y"]) for _ in range(3)]
        bases_history.append(bases)
        if bases.count("Y") % 2 == 0:
            circuits[i] = create_circuit_interception(bases)

    # Table Printing
    cols = ["Round", "Bases", "Keep"]
    widths = [5, 5, 4]
    print_table_header(cols, widths)
    for i in range(TOTAL_ROUNDS):
        print_table_row([i, "".join(bases_history[i]), "Y" if i in circuits else "N"], widths)

    return circuits, bases_history

# ==========================================
# 3. TEST ROUNDS
# ==========================================

def key_gen(circuits, bases_history):
    valid_rounds = list(circuits.keys())
    random.shuffle(valid_rounds)
    total_valid = len(valid_rounds)
    test_count = math.ceil(total_valid * TEST_ROUNDS_PERCENTAGE)
    data_count = total_valid - test_count

    # Check if enough data bits
    if data_count < OTP_LENGTH:
        print(f"\n>>> CRITICAL ERROR: Not enough data rounds to satisfy length.")
        return

    test_rounds_indices = valid_rounds[:test_count]
    data_rounds_indices = valid_rounds[test_count : test_count + OTP_LENGTH]

    # ==========================================
    # 3.1 TEST ROUNDS
    # ==========================================

    print("\n=== 2. TEST ROUNDS ===")

    job_results = {}
    computed_results = {}

    # Circuit Creation
    for i in test_rounds_indices:
        use_as_test_round(circuits[i], bases_history[i])
    test_circuits = [circuits[i] for i in test_rounds_indices]

    # Run on Backend
    print("-> Transpiling and sending test rounds job...")

    if use_lagrange == True:
        transpiled = transpile(test_circuits, backend, initial_layout=QUBIT_LAYOUT,
                               optimization_level=3)
    else:
        transpiled = transpile(test_circuits, backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # Charlie* Postprocessing
    print("-> Results received.")

    results = job.result()
    for i in range(len(test_rounds_indices)):
        counts = results.get_counts(i)
        raw_outcome = majority(counts)
        # Reverses to: a b c e
        job_results[test_rounds_indices[i]] = raw_outcome[::-1]

    for i in test_rounds_indices:
        outcome = job_results[i][:2]
        charlie_outcome = job_results[i][2:]
        bases = "".join(bases_history[i])[:2]
        if bases == "XX":
            if   charlie_outcome == "10" or charlie_outcome == "01": outcome += "0"
            elif charlie_outcome == "00" or charlie_outcome == "11": outcome += "1"
        elif bases == "XY":
            if   charlie_outcome == "10" or charlie_outcome == "11": outcome += "0"
            elif charlie_outcome == "00" or charlie_outcome == "01": outcome += "1"
        elif bases == "YX":
            if   charlie_outcome == "10" or charlie_outcome == "01": outcome += "0"
            elif charlie_outcome == "00" or charlie_outcome == "11": outcome += "1"
        elif bases == "YY":
            if   charlie_outcome == "10" or charlie_outcome == "11": outcome += "0"
            elif charlie_outcome == "00" or charlie_outcome == "01": outcome += "1"
        computed_results[i] = outcome

    # Table Printing
    cols = ["Round", "Bases", "Result", "Computed", "Measured", "Expected"]
    widths = [5, 6, 8, 8, 8, 8]
    print_table_header(cols, widths)
    good_rounds = 0
    for i in test_rounds_indices:
        measured_parity = get_bitstring_parity(computed_results[i])
        expected_parity = get_bases_parity(bases_history[i])
        print_table_row([i, "".join(bases_history[i]), job_results[i], computed_results[i],
                         measured_parity, expected_parity], widths)

        if measured_parity == expected_parity:
            good_rounds += 1
    error_rate = (1 - good_rounds / test_count)

    print(f"\n-> Error Rate: {error_rate:.2%}")

    if error_rate >= ERROR_THRESHOLD:
        print(f"\n>>> CRITICAL ERROR: Potential eavesdropping detected.")
        return

    # ==========================================
    # 3.2 TEST ROUNDS
    # ==========================================

    print("\n=== 3. DATA ROUNDS ===")
    print(f"-> Selected Rounds: {data_rounds_indices}")

    outcomes = {}

    # Circuit Creation
    for i in data_rounds_indices:
        use_as_data_round(circuits[i], bases_history[i])
    data_circuits = [circuits[i] for i in data_rounds_indices]

    # Run on Backend
    print("-> Transpiling and sending data rounds job...")

    if use_lagrange == True:
        transpiled = transpile(data_circuits, backend, initial_layout=QUBIT_LAYOUT, optimization_level=3)
    else:
        transpiled = transpile(data_circuits, backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # Charlie* Postprocessing
    print("-> Results received.\n")

    results = job.result()
    for i in range(len(data_rounds_indices)):
        counts = results.get_counts(i)
        raw_outcome = majority(counts)
        # Reverses to: a b c e
        outcomes[data_rounds_indices[i]] = raw_outcome[::-1]

    alice_key = ""
    deduced_key = ""

    for outcome in outcomes.values():
        alice_key += outcome[0]

    for i in data_rounds_indices:
        charlie_result = outcomes[i][2:]
        bases = "".join(bases_history[i])[:2]
        if bases == "XX":
            if   charlie_result == "00" or charlie_result == "11": deduced_key += "0"
            elif charlie_result == "10" or charlie_result == "01": deduced_key += "1"
        elif bases == "XY":
            if   charlie_result == "00" or charlie_result == "11": deduced_key += "0"
            elif charlie_result == "10" or charlie_result == "01": deduced_key += "1"
        elif bases == "YX":
            if   charlie_result == "10" or charlie_result == "01": deduced_key += "0"
            elif charlie_result == "00" or charlie_result == "11": deduced_key += "1"
        elif bases == "YY":
            if   charlie_result == "10" or charlie_result == "01": deduced_key += "0"
            elif charlie_result == "00" or charlie_result == "11": deduced_key += "1"

    print(f"-> Alice's Key:  {alice_key}")
    print(f"-> Deduced Key:  {deduced_key}")

    if alice_key == deduced_key:
        print("\n>>> SUCCESS: Key deduced perfectly!")
    else:
        print("\n>>> FAILURE: Mismatch detected.")

# ==========================================
# 4. MAIN ENTRY POINT
# ==========================================

if __name__ == "__main__":
    try:
        circuits, bases_history = sifting()
        key_gen(circuits, bases_history)

    except Exception as e:
        print(f"An error occurred: {e}")


=== 1. INITIALIZATION ===
------------------------
| Round | Bases | Keep |
------------------------
| 0     | XYX   | N    |
| 1     | XYX   | N    |
| 2     | XXY   | N    |
| 3     | YYX   | Y    |
| 4     | XXX   | Y    |
| 5     | YXY   | Y    |
| 6     | XXX   | Y    |
| 7     | XXY   | N    |
| 8     | YYY   | N    |
| 9     | XYX   | N    |
| 10    | XXX   | Y    |
| 11    | XYY   | Y    |
| 12    | XYY   | Y    |
| 13    | YYX   | Y    |
| 14    | XYY   | Y    |
| 15    | XXY   | N    |
| 16    | YXX   | N    |
| 17    | XXX   | Y    |
| 18    | XYX   | N    |
| 19    | XXX   | Y    |
| 20    | XXY   | N    |
| 21    | XXY   | N    |
| 22    | YXY   | Y    |
| 23    | XYY   | Y    |
| 24    | YYX   | Y    |
| 25    | XXY   | N    |
| 26    | YYX   | Y    |
| 27    | YYY   | N    |
| 28    | YYY   | N    |
| 29    | YXY   | Y    |
| 30    | YYY   | N    |
| 31    | XXX   | Y    |
| 32    | XXY   | N    |
| 33    | YXX   | N    |
| 34    | YYY   | N    |
| 35    | XXY   | N    

# Demo 5: Fairness Fix in Action

## Fake Entanglement Attack

In [ ]:
# ==========================================
# 0. CONFIGURATION
# ==========================================

GHZ_QUBIT_LAYOUT = [0, 2, 1]  # Physical mapping: [a, b, c]
BS_QUBIT_LAYOUT = [0, 2]  # Physical mapping: [b', c']

# ==========================================
# 1. CIRCUITS
# ==========================================

def create_ghz(bases):
    ghz = QuantumCircuit(3, 3)

    # State Preparation
    ghz.h(0)
    ghz.cx(0, 1)
    ghz.cx(1, 2)
    ghz.barrier()

    # Alice's Measurement
    if bases[0] == "Y": ghz.sdg(0)
    ghz.h(0)
    ghz.measure(0, 0)
    ghz.barrier()

    # Charlie*'s Measurements
    if random.choice(["X", "Y"]) == "X":
        if bases[2] == "Y": ghz.sdg([1, 2]);
    else:
        if bases[2] == "X":   ghz.sdg(1);
        elif bases[2] == "Y": ghz.sdg(2);
    ghz.h([1, 2])
    ghz.measure([1, 2], [1, 2])

    return ghz

def create_bs(bases):
    bs = QuantumCircuit(2, 2)

    # State Preparation
    bs.x(0)
    bs.h(0)
    bs.cx(0, 1)
    bs.barrier()

    # Bob's Measurement
    if bases[1] == "Y": bs.sdg(0)
    bs.h(0)
    bs.measure(0, 0)
    bs.barrier()

    # Charlie*'s Measurement
    if random.choice(["X", "Y"]) == "Y": bs.sdg(1)
    bs.h(1)
    bs.measure(1, 1)

    return bs

# ==========================================
# 2. PROTOCOL
# ==========================================

def run_protocol(backend):
    print("\n=== 1. INITIALIZATION & EXECUTION ===")

    ghz_circuits = {}
    bs_circuits = {}
    ghz_results = {}
    bs_results = {}
    bases_history = []

    # Circuits Creation
    for i in range(TOTAL_ROUNDS):
        bases = [random.choice(["X", "Y"]) for _ in range(3)]
        bases_history.append(bases)
        if bases.count("Y") % 2 == 0:
            ghz_circuits[i] = create_ghz(bases)
            bs_circuits[i] = create_bs(bases)

    # GHZ States

    # -Run on Backend
    print("-> Transpiling and sending GHZ job...")

    if use_lagrange == True:
        transpiled = transpile(list(ghz_circuits.values()), backend, initial_layout=GHZ_QUBIT_LAYOUT,
                           optimization_level=3)
    else:
        transpiled = transpile(list(ghz_circuits.values()), backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=GHZ_QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # -Process Results
    print("-> GHZ results received.")

    result = job.result()
    for idx, round_id in enumerate(ghz_circuits.keys()):
        counts = result.get_counts(idx)
        raw_outcome = majority(counts)
        # Reverses to: a b c
        ghz_results[round_id] = raw_outcome[::-1]

    # Bell States

    # -Run on Backend
    print("-> Transpiling and sending BS job...")

    if use_lagrange == True:
        transpiled = transpile(list(bs_circuits.values()), backend, initial_layout=BS_QUBIT_LAYOUT,
                            optimization_level=3)
    else:
        transpiled = transpile(list(bs_circuits.values()), backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=BS_QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # -Process Results
    print("-> BS results received.")

    result = job.result()
    for idx, round_id in enumerate(bs_circuits.keys()):
        counts = result.get_counts(idx)
        raw_outcome = majority(counts)
        # Reverses to: b' c'
        bs_results[round_id] = raw_outcome[::-1]


    # Table Printing
    cols = ["Round", "Bases", "GHZ State", "Bell State"]
    widths = [5, 5, 9, 10]
    print_table_header(cols, widths)
    for i in range(TOTAL_ROUNDS):
        if i in bs_results:
            print_table_row([i, "".join(bases_history[i]), ghz_results[i], bs_results[i]], widths)
        else:
            print_table_row([i, "".join(bases_history[i]), "-", "-"], widths)

    return ghz_results, bs_results, bases_history

# ==========================================
# 3. SIDE FUNCTIONS
# ==========================================

def get_alice_bit(ghz_meas, bases):
    used_bases = []
    if bases[0] == "X":
        used_bases.append("X")
        if bases[2] == "X":   used_bases.append("X"); used_bases.append("X")
        elif bases[2] == "Y": used_bases.append("Y"); used_bases.append("Y")
    elif bases[0] == "Y":
        used_bases.append("Y")
        if bases[2] == "X":   used_bases.append("Y"); used_bases.append("X")
        elif bases[2] == "Y": used_bases.append("X"); used_bases.append("Y")
    alice_bit = str(get_bitstring_parity(ghz_meas[1] + ghz_meas[2] + str(get_bases_parity(used_bases))))

    return alice_bit

def get_bob_bit(bs_meas, bases):
    bob_bit = ""
    charlie_bit = bs_meas[1]
    if bases[1] == "X":
        if charlie_bit == "0":   bob_bit = "1"
        elif charlie_bit == "1": bob_bit = "0"
    elif bases[1] == "Y":
        if charlie_bit == "0":   bob_bit = "0"
        elif charlie_bit == "1": bob_bit = "1"

    return bob_bit

def get_expected_bit(ghz_meas, bs_meas, bases):
    alice_bit = get_alice_bit(ghz_meas, bases)
    bob_bit = get_bob_bit(bs_meas, bases)
    expected_bit = str(alice_bit + bob_bit + str(get_bases_parity(bases)))

    return expected_bit

# ==========================================
# 4. KEY GEN
# ==========================================

def key_gen(ghz_results, bs_results, bases_history):
    valid_rounds = list(ghz_results.keys())
    random.shuffle(valid_rounds)
    total_valid = len(valid_rounds)
    test_count = math.ceil(total_valid * TEST_ROUNDS_PERCENTAGE)
    data_count = total_valid - test_count

    # Check if enough data bits
    if data_count < OTP_LENGTH:
        print(f"\n>>> CRITICAL ERROR: Not enough data rounds to satisfy length.")
        return

    test_rounds_indices = valid_rounds[:test_count]
    data_rounds_indices = valid_rounds[test_count : test_count + OTP_LENGTH]

    # ==========================================
    # 4.1 TEST ROUNDS
    # ==========================================

    print("\n=== 2. TEST ROUNDS ===")

    cols = ["Round", "Expected", "Measured"]
    widths = [5, 8, 8]
    print_table_header(cols, widths)
    good_rounds = 0
    for i in test_rounds_indices:
        final_result = ghz_results[i][0] \
                     + bs_results[i][0]  \
                     + get_expected_bit(ghz_results[i], bs_results[i], bases_history[i])
        measured_parity = get_bitstring_parity(final_result)
        expected_parity = get_bases_parity(bases_history[i])
        print_table_row([i, measured_parity, expected_parity], widths)

        if measured_parity == expected_parity:
            good_rounds += 1
    error_rate = (1 - good_rounds / test_count)

    print(f"\n-> Error Rate: {error_rate:.2%}")

    if error_rate >= ERROR_THRESHOLD:
        print(f"\n>>> CRITICAL ERROR: Potential eavesdropping detected.")
        return

    # ==========================================
    # 4.2 DATA ROUNDS
    # ==========================================

    print("\n=== 3. DATA ROUNDS ===")
    print(f"-> Selected Rounds: {data_rounds_indices}\n")

    alice_key = ""
    deduced_key = ""

    for i in data_rounds_indices:
        alice_key += ghz_results[i][0]
        deduced_key += get_alice_bit(ghz_results[i], bases_history[i])

    print(f"-> Alice's Key:  {alice_key}")
    print(f"-> Deduced Key:  {deduced_key}")

    if alice_key == deduced_key:
        print("\n>>> SUCCESS: Key deduced perfectly!")
    else:
        print("\n>>> FAILURE: Mismatch detected.")

# ==========================================
# 5. MAIN
# ==========================================

if __name__ == "__main__":
    try:
        ghz_results, bs_results, bases_history = run_protocol(backend)
        key_gen(ghz_results, bs_results, bases_history)

    except Exception as e:
        print(f"An error occurred: {e}")


=== 1. INITIALIZATION & EXECUTION ===
-> Transpiling and sending GHZ job...
-> GHZ results received.
-> Transpiling and sending BS job...
-> BS results received.
------------------------------------------
| Round | Bases | GHZ State | Bell State |
------------------------------------------
| 0     | YXY   | 001       | 10         |
| 1     | YYX   | 000       | 11         |
| 2     | YXX   | -         | -          |
| 3     | XYX   | -         | -          |
| 4     | YYY   | -         | -          |
| 5     | XYY   | 010       | 00         |
| 6     | XYX   | -         | -          |
| 7     | YYY   | -         | -          |
| 8     | YYX   | 011       | 11         |
| 9     | XYY   | 000       | 10         |
| 10    | YXY   | 010       | 01         |
| 11    | XYY   | 011       | 11         |
| 12    | YYX   | 011       | 11         |
| 13    | XYY   | 111       | 01         |
| 14    | YXY   | 001       | 01         |
| 15    | XXY   | -         | -          |
| 16    | YYY   | - 

## Ancilla Attack

In [ ]:
# ==========================================
# 0. CONFIGURATION
# ==========================================

QUBIT_LAYOUT = [0, 2, 1, 3]  # Physical mapping: [Alice, Bob, Charlie, Ancilla]

# ==========================================
# 1. CIRCUITS
# ==========================================

def create_circuit_interception(bases):
    qc = QuantumCircuit(4, 4)

    # GHZ State Preparation
    qc.h(0)
    qc.cx(0, 1)
    qc.cx(1, 2)
    qc.barrier()

    # Charlie*'s Interception
    qc.h(1)
    qc.cx(1, 3)
    qc.barrier()

    # Alice's and Bob's Measurements
    for i, basis in enumerate(bases[:2]):
        if basis == "Y": qc.sdg(i)
        qc.h(i)
        qc.measure(i, i)
    qc.barrier()

    return qc

def use_as_test_round(circuit):
    guessed_bases = [random.choice(["X", "Y"]) for _ in range(2)]
    circuit.cx(2, 3)
    if guessed_bases.count("Y") == 1: circuit.sdg(2)
    circuit.h(2)
    circuit.measure([2, 3], [2, 3])

def use_as_data_round(circuit):
    if random.choice(["X", "Y"]) == "Y": circuit.sdg(2)
    circuit.h(2)
    circuit.measure([2, 3], [2, 3])


# ==========================================
# 2. PROTOCOL
# ==========================================

def sifting():
    print("\n=== 1. INITIALIZATION ===")

    circuits = {}
    bases_history = []

    # Sifting
    for i in range(TOTAL_ROUNDS):
        bases = [random.choice(["X", "Y"]) for _ in range(3)]
        bases_history.append(bases)
        if bases.count("Y") % 2 == 0:
            circuits[i] = create_circuit_interception(bases)

    # Table Printing
    cols = ["Round", "Bases", "Keep"]
    widths = [5, 5, 4]
    print_table_header(cols, widths)
    for i in range(TOTAL_ROUNDS):
        print_table_row([i, "".join(bases_history[i]), "Y" if i in circuits else "N"], widths)

    return circuits, bases_history

# ==========================================
# 3. TEST ROUNDS
# ==========================================

def key_gen(circuits, bases_history):
    valid_rounds = list(circuits.keys())
    random.shuffle(valid_rounds)
    total_valid = len(valid_rounds)
    test_count = math.ceil(total_valid * TEST_ROUNDS_PERCENTAGE)
    data_count = total_valid - test_count

    # Check if enough data bits
    if data_count < OTP_LENGTH:
        print(f"\n>>> CRITICAL ERROR: Not enough data rounds to satisfy length.")
        return

    test_rounds_indices = valid_rounds[:test_count]
    data_rounds_indices = valid_rounds[test_count : test_count + OTP_LENGTH]

    # ==========================================
    # 3.1 TEST ROUNDS
    # ==========================================

    print("\n=== 2. TEST ROUNDS ===")

    job_results = {}
    computed_results = {}

    # Circuit Creation
    for i in test_rounds_indices:
        use_as_test_round(circuits[i])
    test_circuits = [circuits[i] for i in test_rounds_indices]

    # Run on Backend
    print("-> Transpiling and sending test rounds job...")

    if use_lagrange == True:
        transpiled = transpile(test_circuits, backend, initial_layout=QUBIT_LAYOUT,
                               optimization_level=3)
    else:
        transpiled = transpile(test_circuits, backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # Charlie* Postprocessing
    print("-> Results received.")

    results = job.result()
    for i in range(len(test_rounds_indices)):
        counts = results.get_counts(i)
        raw_outcome = majority(counts)
        # Reverses to: a b c e
        job_results[test_rounds_indices[i]] = raw_outcome[::-1]

    for i in test_rounds_indices:
        outcome = job_results[i][:2]
        charlie_outcome = job_results[i][2:]
        bases = "".join(bases_history[i])[:2]
        if bases == "XX":
            if   charlie_outcome == "10" or charlie_outcome == "01": outcome += "0"
            elif charlie_outcome == "00" or charlie_outcome == "11": outcome += "1"
        elif bases == "XY":
            if   charlie_outcome == "10" or charlie_outcome == "11": outcome += "0"
            elif charlie_outcome == "00" or charlie_outcome == "01": outcome += "1"
        elif bases == "YX":
            if   charlie_outcome == "10" or charlie_outcome == "01": outcome += "0"
            elif charlie_outcome == "00" or charlie_outcome == "11": outcome += "1"
        elif bases == "YY":
            if   charlie_outcome == "10" or charlie_outcome == "11": outcome += "0"
            elif charlie_outcome == "00" or charlie_outcome == "01": outcome += "1"
        computed_results[i] = outcome

    # Table Printing
    cols = ["Round", "Bases", "Result", "Computed", "Measured", "Expected"]
    widths = [5, 6, 8, 8, 8, 8]
    print_table_header(cols, widths)
    good_rounds = 0
    for i in test_rounds_indices:
        measured_parity = get_bitstring_parity(computed_results[i])
        expected_parity = get_bases_parity(bases_history[i])
        print_table_row([i, "".join(bases_history[i]), job_results[i], computed_results[i],
                         expected_parity, measured_parity], widths)

        if measured_parity == expected_parity:
            good_rounds += 1
    error_rate = (1 - good_rounds / test_count)

    print(f"\n-> Error Rate: {error_rate:.2%}")

    if error_rate >= ERROR_THRESHOLD:
        print(f"\n>>> CRITICAL ERROR: Potential eavesdropping detected.")
        return

    # ==========================================
    # 3.2 TEST ROUNDS
    # ==========================================

    print("\n=== 3. DATA ROUNDS ===")
    print(f"-> Selected Rounds: {data_rounds_indices}")

    outcomes = {}

    # Circuit Creation
    for i in data_rounds_indices:
        use_as_data_round(circuits[i])
    data_circuits = [circuits[i] for i in data_rounds_indices]

    # Run on Backend
    print("-> Transpiling and sending data rounds job...")

    if use_lagrange == True:
        transpiled = transpile(data_circuits, backend, initial_layout=QUBIT_LAYOUT, optimization_level=3)
    else:
        transpiled = transpile(data_circuits, backend, basis_gates=iqm_basis_gates,
                               coupling_map=spark_coupling_map, initial_layout=QUBIT_LAYOUT,
                               optimization_level=3)
    job = backend.run(transpiled, shots=SHOTS_PER_CIRCUIT)

    # Charlie* Postprocessing
    print("-> Results received.\n")

    results = job.result()
    for i in range(len(data_rounds_indices)):
        counts = results.get_counts(i)
        raw_outcome = majority(counts)
        # Reverses to: a b c e
        outcomes[data_rounds_indices[i]] = raw_outcome[::-1]

    alice_key = ""
    deduced_key = ""

    for outcome in outcomes.values():
        alice_key += outcome[0]

    for i in data_rounds_indices:
        charlie_result = outcomes[i][2:]
        bases = "".join(bases_history[i])[:2]
        if bases == "XX":
            if   charlie_result == "00" or charlie_result == "11": deduced_key += "0"
            elif charlie_result == "10" or charlie_result == "01": deduced_key += "1"
        elif bases == "XY":
            if   charlie_result == "00" or charlie_result == "11": deduced_key += "0"
            elif charlie_result == "10" or charlie_result == "01": deduced_key += "1"
        elif bases == "YX":
            if   charlie_result == "10" or charlie_result == "01": deduced_key += "0"
            elif charlie_result == "00" or charlie_result == "11": deduced_key += "1"
        elif bases == "YY":
            if   charlie_result == "10" or charlie_result == "01": deduced_key += "0"
            elif charlie_result == "00" or charlie_result == "11": deduced_key += "1"

    print(f"-> Alice's Key:  {alice_key}")
    print(f"-> Deduced Key:  {deduced_key}")

    if alice_key == deduced_key:
        print("\n>>> SUCCESS: Key deduced perfectly!")
    else:
        print("\n>>> FAILURE: Mismatch detected.")

# ==========================================
# 4. MAIN ENTRY POINT
# ==========================================

if __name__ == "__main__":
    try:
        circuits, bases_history = sifting()
        key_gen(circuits, bases_history)

    except Exception as e:
        print(f"An error occurred: {e}")


=== 1. INITIALIZATION ===
------------------------
| Round | Bases | Keep |
------------------------
| 0     | YYX   | Y    |
| 1     | YXY   | Y    |
| 2     | XXY   | N    |
| 3     | XYX   | N    |
| 4     | YXX   | N    |
| 5     | YXY   | Y    |
| 6     | XXY   | N    |
| 7     | YXY   | Y    |
| 8     | XXY   | N    |
| 9     | YYX   | Y    |
| 10    | YXX   | N    |
| 11    | YYY   | N    |
| 12    | XXY   | N    |
| 13    | YYY   | N    |
| 14    | YYX   | Y    |
| 15    | YYX   | Y    |
| 16    | XYX   | N    |
| 17    | XYY   | Y    |
| 18    | XXY   | N    |
| 19    | XXY   | N    |
| 20    | XXX   | Y    |
| 21    | XYX   | N    |
| 22    | XYY   | Y    |
| 23    | YYY   | N    |
| 24    | XXY   | N    |
| 25    | XXX   | Y    |
| 26    | YYX   | Y    |
| 27    | YXX   | N    |
| 28    | XXY   | N    |
| 29    | YXY   | Y    |
| 30    | XYX   | N    |
| 31    | XXY   | N    |
| 32    | YYY   | N    |
| 33    | XYY   | Y    |
| 34    | YXY   | Y    |
| 35    | XYY   | Y    